In [11]:
import json
import nltk
from nltk.corpus import wordnet as wn
from nltk import PorterStemmer
from tqdm import tqdm
import pickle

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\deevy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [12]:
stemmer = PorterStemmer()

In [13]:
with open('data/hotpotqa_vocab_term_to_id.json') as f:
    term_to_id = json.load(f)
with open('data/hotpotqa_vocab_id_to_term.json') as f:
    id_to_term = json.load(f)

In [14]:
vocab = list(term_to_id.keys())
len(vocab)

45042

In [15]:
stem_to_lemmas = {}
for synset in wn.all_synsets():
    for lemma in synset.lemma_names():
        stem = stemmer.stem(lemma)
        stem_to_lemmas.setdefault(stem, set()).add(lemma)

In [16]:
def expand_stemmed_vocab(stem, vocab_stems, stem_to_lemmas, stemmer):
    expanded = set()

    for lemma in stem_to_lemmas.get(stem, []):
        for syn in wn.synsets(lemma):
            for l in syn.lemmas():
                w = l.name()
                s = stemmer.stem(w)
                if s in vocab_stems:
                    expanded.add(s)
            for h in syn.hypernyms():
                for l in h.lemmas():
                    w = l.name()
                    s = stemmer.stem(w)
                    if s in vocab_stems:
                        expanded.add(s)
            for h in syn.hyponyms():
                for l in h.lemmas():
                    w = l.name()
                    s = stemmer.stem(w)
                    if s in vocab_stems:
                        expanded.add(s)
        
    return expanded

In [17]:
expanded_stems = expand_stemmed_vocab("comput", vocab, stem_to_lemmas, stemmer)
expanded_stems

{'actuari',
 'adder',
 'averag',
 'budget',
 'calcul',
 'capit',
 'cipher',
 'client',
 'comput',
 'cypher',
 'deduct',
 'differenti',
 'engin',
 'estim',
 'extract',
 'factor',
 'figur',
 'fraction',
 'gaug',
 'guess',
 'host',
 'idea',
 'integr',
 'interpol',
 'judg',
 'machin',
 'multipli',
 'node',
 'object',
 'oper',
 'predictor',
 'procedur',
 'process',
 'quantiz',
 'server',
 'site',
 'statistician',
 'subtract',
 'technolog',
 'transposit',
 'websit'}

In [18]:
all_related_vocab_ids = set()
for stem in tqdm(vocab):
    related_stems = expand_stemmed_vocab(stem, vocab, stem_to_lemmas, stemmer)
    stem_id = term_to_id[stem]
    for related_stem in related_stems:
        related_stem_id = term_to_id[related_stem]
        if related_stem_id > stem_id:
            tuple_to_be_added = (stem_id, related_stem_id)
            tuple_to_be_added_reversed = (related_stem_id, stem_id)
        else:
            tuple_to_be_added = (related_stem_id, stem_id)
            tuple_to_be_added_reversed = (stem_id, related_stem_id)
        if tuple_to_be_added not in all_related_vocab_ids:
            all_related_vocab_ids.add(tuple_to_be_added)
        if tuple_to_be_added_reversed not in all_related_vocab_ids:
            all_related_vocab_ids.add(tuple_to_be_added_reversed)
    if (stem_id, stem_id) not in all_related_vocab_ids:
        all_related_vocab_ids.add((stem_id, stem_id))

100%|██████████| 45042/45042 [03:01<00:00, 247.63it/s] 


In [19]:
len(all_related_vocab_ids)

156646

In [20]:
with open('data/wordnet_pairs.pkl', 'wb') as f:
    pickle.dump(all_related_vocab_ids, f)